# Heart Disease Downstream (Notebook 2)

This notebook starts after all encoding work is done. It loads the artifacts created by `01_build_heart_failure_artifacts`, then trains static-only heart-failure classifiers.

## What changed versus `02_train_evaluate_models`
- No retrieval
- No day embeddings
- No risk trajectory features
- Task is now heart failure present during admission, defined from `428*`
- Inputs are just the saved day-0 static embeddings

## What this notebook does
1. Loads the saved metadata and static embeddings
2. Fits a logistic-regression baseline
3. Fits a small MLP baseline
4. Evaluates AUROC and AUPRC on validation and test
5. Saves models, predictions, and summaries

## Note: as described in README, MIMIC-III is not included in this project, authorised PhysioNet access is required.

In [ ]:
# ============================================================
# CELL 1: Paths + select experiment folder
# ============================================================

from pathlib import Path

MAX_HOSP_DAY = 0
SEED = 42
EXPERIMENT_NAME = f"your heart disease experiment name"

ARTIFACT_ROOT = Path("your heart disease artifact root")
EXP_DIR = ARTIFACT_ROOT / EXPERIMENT_NAME

META_DIR = EXP_DIR / "metadata"
EMB_DIR = EXP_DIR / "embeddings"
MODEL_DIR = EXP_DIR / "models"

print("Loading experiment from:")
print(EXP_DIR)


In [ ]:
# ============================================================
# CELL 2: Imports + reproducibility
# ============================================================

import json
import pickle
import random
from typing import Dict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from IPython.display import display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("DEVICE:", DEVICE)


In [ ]:
# ============================================================
# CELL 3: Helper functions
# ============================================================

def load_npz_array(path: Path) -> np.ndarray:
    return np.load(path)["array"]


def save_pickle(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def evaluate_probs(name: str, y: np.ndarray, probs: np.ndarray) -> Dict[str, float]:
    out = {
        "model": name,
        "AUROC": float(roc_auc_score(y, probs)),
        "AUPRC": float(average_precision_score(y, probs)),
    }
    print(f"{name}: AUROC={out['AUROC']:.4f}  AUPRC={out['AUPRC']:.4f}")
    return out


def evaluate_lr(name: str, clf, X: np.ndarray, y: np.ndarray) -> Dict[str, float]:
    probs = clf.predict_proba(X)[:, 1]
    return evaluate_probs(name, y, probs)


def make_loader(X, y, batch_size=256, shuffle=True):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle, drop_last=False)


class SmallMLP(nn.Module):
    def __init__(self, in_dim, hidden1=256, hidden2=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


@torch.no_grad()
def predict_proba_nn(model, X, batch_size=4096):
    model.eval()
    probs = []
    for start in range(0, len(X), batch_size):
        xb = torch.tensor(X[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        logits = model(xb)
        batch_probs = torch.sigmoid(logits).detach().cpu().numpy()
        probs.append(batch_probs)
    return np.concatenate(probs)


def evaluate_nn(name, model, X, y):
    probs = predict_proba_nn(model, X)
    out = evaluate_probs(name, y, probs)
    return out, probs


def train_mlp(
    X_train, y_train, X_val, y_val,
    hidden1=256, hidden2=64, dropout=0.3,
    lr=1e-3, weight_decay=1e-4, batch_size=256,
    max_epochs=60, patience=8, seed=42
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = SmallMLP(X_train.shape[1], hidden1, hidden2, dropout).to(DEVICE)
    train_loader = make_loader(X_train, y_train, batch_size=batch_size, shuffle=True)

    pos = float(y_train.sum())
    neg = float(len(y_train) - pos)
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_state = None
    best_val_auprc = -1.0
    bad_epochs = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optim.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

        val_probs = predict_proba_nn(model, X_val)
        val_auprc = average_precision_score(y_val, val_probs)

        if val_auprc > best_val_auprc + 1e-4:
            best_val_auprc = val_auprc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


In [ ]:
# ============================================================
# CELL 4: Load manifest, metadata tables, and saved embeddings
# ============================================================

with open(META_DIR / "manifest.json", "r") as f:
    manifest = json.load(f)

print("Manifest loaded:")
print(json.dumps(manifest, indent=2))

if int(manifest["max_hosp_day"]) != int(MAX_HOSP_DAY):
    raise ValueError(
        f"MAX_HOSP_DAY in Notebook 2 ({MAX_HOSP_DAY}) does not match manifest ({manifest['max_hosp_day']})."
    )

train_adm = pd.read_parquet(META_DIR / "train_adm_meta.parquet")
val_adm   = pd.read_parquet(META_DIR / "val_adm_meta.parquet")
test_adm  = pd.read_parquet(META_DIR / "test_adm_meta.parquet")

X_train_static = load_npz_array(EMB_DIR / "static_train.npz")
X_val_static   = load_npz_array(EMB_DIR / "static_val.npz")
X_test_static  = load_npz_array(EMB_DIR / "static_test.npz")

y_train = train_adm["label"].values.astype(int)
y_val   = val_adm["label"].values.astype(int)
y_test  = test_adm["label"].values.astype(int)

print("\\nShapes:")
print("train_adm:", train_adm.shape, "| X_train_static:", X_train_static.shape)
print("val_adm:  ", val_adm.shape,   "| X_val_static:  ", X_val_static.shape)
print("test_adm: ", test_adm.shape,  "| X_test_static: ", X_test_static.shape)

assert len(train_adm) == len(X_train_static)
assert len(val_adm) == len(X_val_static)
assert len(test_adm) == len(X_test_static)

print("\\nHF prevalence by split:")
print({
    "train": float(y_train.mean()),
    "val": float(y_val.mean()),
    "test": float(y_test.mean()),
})


In [ ]:
# ============================================================
# CELL 5: Standardise the static embeddings
# ============================================================

emb_scaler = StandardScaler(with_mean=True, with_std=True)
X_train_static_nn = emb_scaler.fit_transform(X_train_static)
X_val_static_nn   = emb_scaler.transform(X_val_static)
X_test_static_nn  = emb_scaler.transform(X_test_static)

save_pickle(emb_scaler, MODEL_DIR / "embedding_scaler.pkl")

print("Scaled static feature shapes:")
print("X_train_static_nn:", X_train_static_nn.shape)
print("X_val_static_nn:  ", X_val_static_nn.shape)
print("X_test_static_nn: ", X_test_static_nn.shape)


In [ ]:
# ============================================================
# CELL 6: Fit logistic-regression baseline
# ============================================================

clf_static = LogisticRegression(max_iter=2000, n_jobs=-1, class_weight="balanced")
clf_static.fit(X_train_static, y_train)

lr_results = []
lr_results.append(evaluate_lr("LR static | val", clf_static, X_val_static, y_val))
lr_results.append(evaluate_lr("LR static | test", clf_static, X_test_static, y_test))

save_pickle(clf_static, MODEL_DIR / "clf_static.pkl")
print("Saved clf_static.pkl")


In [ ]:
# ============================================================
# CELL 7: Train and evaluate the MLP baseline
# ============================================================

BATCH_SIZE = 256
HIDDEN1 = 256
HIDDEN2 = 64
DROPOUT = 0.3
LR = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 60
PATIENCE = 8

nn_results = []
prediction_tables = []

mlp_static = train_mlp(
    X_train_static_nn, y_train,
    X_val_static_nn, y_val,
    hidden1=HIDDEN1, hidden2=HIDDEN2, dropout=DROPOUT,
    lr=LR, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, seed=SEED
)

res_val_static, val_probs_static = evaluate_nn("MLP static | val", mlp_static, X_val_static_nn, y_val)
res_test_static, test_probs_static = evaluate_nn("MLP static | test", mlp_static, X_test_static_nn, y_test)

nn_results.extend([res_val_static, res_test_static])

save_pickle(mlp_static.state_dict(), MODEL_DIR / "mlp_static_state.pkl")
print("Saved mlp_static_state.pkl")

prediction_tables.append(
    val_adm[["HADM_ID", "SUBJECT_ID", "label"]].assign(split="val", model="lr_static", prob=clf_static.predict_proba(X_val_static)[:, 1])
)
prediction_tables.append(
    test_adm[["HADM_ID", "SUBJECT_ID", "label"]].assign(split="test", model="lr_static", prob=clf_static.predict_proba(X_test_static)[:, 1])
)
prediction_tables.append(
    val_adm[["HADM_ID", "SUBJECT_ID", "label"]].assign(split="val", model="mlp_static", prob=val_probs_static)
)
prediction_tables.append(
    test_adm[["HADM_ID", "SUBJECT_ID", "label"]].assign(split="test", model="mlp_static", prob=test_probs_static)
)


In [ ]:
# ============================================================
# CELL 8: Save evaluation summaries and predictions
# ============================================================

lr_results_df = pd.DataFrame(lr_results)
nn_results_df = pd.DataFrame(nn_results)
all_predictions_df = pd.concat(prediction_tables, ignore_index=True)

lr_results_df.to_csv(MODEL_DIR / "lr_results.csv", index=False)
nn_results_df.to_csv(MODEL_DIR / "nn_results.csv", index=False)
all_predictions_df.to_parquet(MODEL_DIR / "all_predictions.parquet", index=False)

summary_rows = []
for model_name, split_name, probs, labels in [
    ("lr_static", "val", clf_static.predict_proba(X_val_static)[:, 1], y_val),
    ("lr_static", "test", clf_static.predict_proba(X_test_static)[:, 1], y_test),
    ("mlp_static", "val", val_probs_static, y_val),
    ("mlp_static", "test", test_probs_static, y_test),
]:
    summary_rows.append({
        "model_name": model_name,
        "split": split_name,
        "n": int(len(labels)),
        "prevalence": float(np.mean(labels)),
        "AUROC": float(roc_auc_score(labels, probs)),
        "AUPRC": float(average_precision_score(labels, probs)),
        "max_hosp_day": int(MAX_HOSP_DAY),
        "task_name": manifest["task_name"],
        "label_definition": manifest["label_definition"],
    })

final_summary_df = pd.DataFrame(summary_rows)
final_summary_df.to_csv(MODEL_DIR / "final_summary.csv", index=False)

print("LR results:")
display(lr_results_df)

print("\\nNN results:")
display(nn_results_df)

print("\\nFinal summary:")
display(final_summary_df)


In [ ]:
# ============================================================
# CELL 9: Quick sanity display
# ============================================================

print("Validation preview:")
display(val_adm.head())

print("\\nTest preview:")
display(test_adm.head())

print("\\nNotebook 2 complete.")
print("Models and evaluation files are saved in:")
print(MODEL_DIR)
